In [6]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from scipy import signal
import pyarrow.parquet as pq
from sklearn.model_selection import train_test_split
from collections import Counter
import gc

# =========================================================
# 1. CONFIGURATION ET PRÉPARATION DES DONNÉES
# =========================================================
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

import pyarrow.parquet as pq

# =========================================================
# 1. PRÉPARATION DES DONNÉES (CHARGEMENT PAR PAQUETS)
# =========================================================
path = "C:/Users/Théo Lassale/Desktop/Perso/Stage/Stage_Malmö/IMU_LM_Data/data/merged_dataset/filtered_3_labels.parquet"

WINDOW_SIZE, STRIDE = 50, 25
X, labels = [], []

# On ouvre le fichier sans le charger en RAM
parquet_file = pq.ParquetFile(path)

print(f"Nombre de paquets à traiter : {parquet_file.num_row_groups}")

# On itère sur chaque paquet (Row Group) du fichier Parquet
for i in range(parquet_file.num_row_groups):
    # Chargement d'un petit morceau (uniquement les colonnes utiles)
    chunk = parquet_file.read_row_group(i, columns=[
        "acc_x", "acc_y", "acc_z", "global_activity_id", 
        "dataset", "subject_id", "session_id"
    ]).to_pandas()
    
    # Calcul Magnitude en float32 (gain de place)
    acc_mag = np.sqrt(chunk["acc_x"]**2 + chunk["acc_y"]**2 + chunk["acc_z"]**2).astype(np.float32)
    chunk["acc_mag"] = acc_mag
    
    # On vire les axes bruts immédiatement
    chunk.drop(columns=["acc_x", "acc_y", "acc_z"], inplace=True)
    
    # Fenêtrage sur ce paquet
    # On groupe par session pour ne pas mélanger les données de différents sujets
    for _, group in chunk.groupby(["dataset", "subject_id", "session_id"]):
        sig = group["acc_mag"].values
        acts = group["global_activity_id"].values
        
        if len(sig) < WINDOW_SIZE:
            continue
            
        for j in range(0, len(sig) - WINDOW_SIZE, STRIDE):
            X.append(sig[j:j + WINDOW_SIZE])
            # On prend le label majoritaire dans la fenêtre
            labels.append(Counter(acts[j:j + WINDOW_SIZE]).most_common(1)[0][0])
    
    # Nettoyage manuel de la RAM après chaque paquet
    del chunk
    gc.collect()
    if (i + 1) % 5 == 0:
        print(f"Paquet {i+1}/{parquet_file.num_row_groups} terminé...")

# Conversion finale en tableaux Numpy
X = np.array(X, dtype=np.float32)
labels = np.array(labels)

# Mapping des labels (pour qu'ils commencent à 0)
unique_labels = sorted(np.unique(labels))
label_mapping = {old: new for new, old in enumerate(unique_labels)}
labels = np.array([label_mapping[l] for l in labels])

print(f"Chargement terminé. Nombre de fenêtres : {len(X)}")

# Split & Normalisation
X_temp, X_test, y_temp, y_test = train_test_split(X, labels, test_size=0.10, random_state=SEED, shuffle=True)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.2222, random_state=SEED, shuffle=True)

mean, std = np.mean(X_train, axis=(0, 1)), np.std(X_train, axis=(0, 1))
X_train_t = torch.tensor(X_train, dtype=torch.float32).unsqueeze(-1).transpose(1, 2)
X_val_t   = torch.tensor(X_val, dtype=torch.float32).unsqueeze(-1).transpose(1, 2)

train_ds = TensorDataset(X_train_t, torch.tensor(y_train, dtype=torch.long))
val_ds   = TensorDataset(X_val_t, torch.tensor(y_val, dtype=torch.long))

train_loader = DataLoader(train_ds, batch_size=8192, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=8192)

# =========================================================
# 2. DÉFINITION DES MODÈLES (VAE + JUGES)
# =========================================================

class VAE(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 32, 5, padding=2), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, 3, padding=1), nn.ReLU(), nn.Flatten()
        )
        self.fc_mu = nn.Linear(128 * 12, latent_dim)
        self.fc_logvar = nn.Linear(128 * 12, latent_dim)
        self.dec_input = nn.Linear(latent_dim, 128 * 13)
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(128, 64, 3, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose1d(64, 32, 5, stride=2, padding=2, output_padding=1), nn.ReLU(),
            nn.Conv1d(32, 1, 5, padding=2)
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def forward(self, x):
        h = self.encoder(x)
        mu, logvar = self.fc_mu(h), self.fc_logvar(h)
        z = self.reparameterize(mu, logvar)
        return self.decoder(self.dec_input(z).view(-1, 128, 13)), mu, logvar

class DeepConvLSTM(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv1d(1, 64, 5, padding=2), nn.ReLU(), nn.Conv1d(64, 64, 5, padding=2), nn.ReLU())
        self.lstm = nn.LSTM(64, 128, num_layers=2, batch_first=True)
        self.fc = nn.Sequential(nn.Linear(128, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, n_classes))
    def forward(self, x):
        x = self.conv(x).transpose(1, 2)
        x, _ = self.lstm(x)
        return self.fc(x[:, -1, :])

class CNNSimple(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.net = nn.Sequential(nn.Conv1d(1, 64, 3, padding=1), nn.ReLU(), nn.Conv1d(64, 64, 3, padding=1), nn.ReLU(),
                                 nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, n_classes))
    def forward(self, x): return self.net(x)

class MLPSimple(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(50, 256), nn.ReLU(), nn.Dropout(0.3),
                                 nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, n_classes))
    def forward(self, x): return self.net(x)

# =========================================================
# 3. ENTRAÎNEMENT
# =========================================================

def train_vae(model, loader, epochs=50):
    opt = optim.Adam(model.parameters(), lr=1e-3)
    model.train()
    for ep in range(epochs):
        total_loss = 0
        for xb, _ in loader:
            xb = xb.to(device)
            opt.zero_grad()
            recon, mu, logvar = model(xb)
            mse = nn.functional.mse_loss(recon, xb, reduction='sum')
            kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
            loss = mse + 0.01 * kld
            loss.backward(); opt.step()
            total_loss += loss.item()
        print(f"VAE Epoch {ep+1} | Loss: {total_loss/len(loader.dataset):.4f}")

def train_judge(model, loader, epochs=30):
    opt = optim.Adam(model.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss()
    model.train()
    for ep in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            crit(model(xb), yb).backward(); opt.step()

vae_model = VAE().to(device)
train_vae(vae_model, train_loader)

judges = [DeepConvLSTM(len(unique_labels)).to(device), 
          CNNSimple(len(unique_labels)).to(device), 
          MLPSimple(len(unique_labels)).to(device)]

for j in judges:
    print(f"Entraînement Juge : {j.__class__.__name__}")
    train_judge(j, train_loader)

# =========================================================
# 4. ANALYSE ET VALIDATION
# =========================================================

def analyse_gait_physique(sig, fs=25):
    mag = sig.flatten()
    mag -= np.mean(mag)
    corr = np.correlate(mag, mag, mode='full')[len(mag)-1:]
    corr /= (np.max(corr) + 1e-7)
    min_l, max_l = int(0.4 * fs), int(1.2 * fs)
    if len(corr) > max_l:
        p_idx = np.argmax(corr[min_l:max_l]) + min_l
        return corr[p_idx], p_idx / fs
    return 0, 1e-7

def full_expert_validation(old_label_id, nom_activite):
    vae_model.eval()
    inv_map = {v: k for k, v in label_mapping.items()}
    
    with torch.no_grad():
        z = torch.randn(30, 32).to(device)
        gen_tensor = vae_model.decoder(vae_model.dec_input(z).view(-1, 128, 13))
        gen_np = gen_tensor.cpu().numpy()

    full_sig = gen_np.flatten()
    score_gait, step_t = analyse_gait_physique(gen_np[0, 0, :])
    f, t_spec, Sxx = signal.spectrogram(full_sig, fs=25)

    fig = plt.figure(figsize=(15, 10))
    plt.subplot(3, 1, 1); plt.plot(full_sig[:1000]); plt.title(f"Génération : {nom_activite}")
    plt.subplot(3, 2, 3); plt.plot(full_sig[250:375], marker='.'); plt.title("Zoom")
    plt.subplot(3, 2, 4); plt.pcolormesh(t_spec, f, 10*np.log10(Sxx+1e-7), shading='gouraud'); plt.title("Spectre")

    ax_txt = plt.subplot(3, 1, 3); ax_txt.axis('off')
    txt = f"VERDICTS JUGES :\n"
    for j in judges:
        j.eval()
        with torch.no_grad():
            p = torch.softmax(j(gen_tensor), dim=1).mean(0)
            c_old = inv_map[torch.argmax(p).item()]
            txt += f"- {j.__class__.__name__}: Label {c_old} | Conf {p.max()*100:.1f}% | {'✅' if c_old==old_label_id else '❌'}\n"
    txt += f"\nGAIT: {score_gait:.2f} | Cadence: {60/step_t:.1f} steps/min"
    ax_txt.text(0, 0.5, txt, fontsize=12, family='monospace')
    plt.tight_layout(); plt.show()

full_expert_validation(1, "Marche")

Nombre de paquets à traiter : 650
Paquet 5/650 terminé...
Paquet 10/650 terminé...
Paquet 15/650 terminé...
Paquet 20/650 terminé...
Paquet 25/650 terminé...
Paquet 30/650 terminé...
Paquet 35/650 terminé...
Paquet 40/650 terminé...
Paquet 45/650 terminé...
Paquet 50/650 terminé...
Paquet 55/650 terminé...
Paquet 60/650 terminé...
Paquet 65/650 terminé...
Paquet 70/650 terminé...
Paquet 75/650 terminé...
Paquet 80/650 terminé...
Paquet 85/650 terminé...
Paquet 90/650 terminé...
Paquet 95/650 terminé...
Paquet 100/650 terminé...
Paquet 105/650 terminé...
Paquet 110/650 terminé...
Paquet 115/650 terminé...
Paquet 120/650 terminé...
Paquet 125/650 terminé...
Paquet 130/650 terminé...
Paquet 135/650 terminé...
Paquet 140/650 terminé...
Paquet 145/650 terminé...
Paquet 150/650 terminé...
Paquet 155/650 terminé...
Paquet 160/650 terminé...
Paquet 165/650 terminé...
Paquet 170/650 terminé...
Paquet 175/650 terminé...
Paquet 180/650 terminé...
Paquet 185/650 terminé...
Paquet 190/650 terminé..

KeyboardInterrupt: 